In [46]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="tanisha",
    database="mini_warehouse"
)

cursor = conn.cursor()
import numpy as np

In [47]:
customers = pd.read_csv(r"C:\Users\Hp\Downloads\archive (1)\customers.csv")

In [48]:
import os 
path1= r"C:\Users\Hp\Downloads\archive (2)\validation\validation"
print(os.listdir(path1))

['bags', 'calendar-and-planning', 'Card-and-baord-Games', 'fashion']


In [49]:
import os
path2 = r"C:\Users\Hp\Downloads\archive (2)\train"
print(os.listdir(path2))

['train']


In [50]:
orders = pd.read_csv(r"C:\Users\Hp\Downloads\archive\orders.csv")

In [51]:
customers.drop_duplicates(inplace=True)
orders.drop_duplicates(inplace=True)

In [52]:
customers.dropna(inplace=True)

In [53]:
orders.dropna(inplace=True)

In [54]:
if 'total_price' in orders.columns:
    orders['total_price']=orders['quantity']*orders['price']

In [55]:
data = customers.merge(orders, left_on='CustomerID', right_on='cust_id')

In [56]:
print(customers.columns)
print(orders.columns)

Index(['CustomerID', 'FirstName', 'MiddleInitial', 'LastName', 'CityID',
       'Address'],
      dtype='str')
Index(['ord_id', 'cust_id', 'ord_amt', 'prd_cat', 'ord_date', 'ord_time'], dtype='str')


In [57]:
cursor.execute("SELECT DATABASE()")
print(cursor.fetchall())

[('mini_warehouse',)]


In [58]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(100),
    city VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_sales (
    order_id INT PRIMARY KEY,
    customer_id INT,
    product_id VARCHAR(50),
    amount FLOAT,
    order_date DATE
)
""")

conn.commit()
print("Tables created ✅")

Tables created ✅


In [59]:
cursor.execute("SHOW TABLES")
print(cursor.fetchall())

[('dim_customers',), ('fact_sales',)]


In [45]:
for i, row in customers.iterrows():
    name = str(row['FirstName']) + " " + str(row['LastName'])
    city = str(row['CityID'])

    cursor.execute(
        "INSERT INTO dim_customers (customer_id, name, city) VALUES (%s, %s, %s)",
        (row['CustomerID'], name, city)
    )

conn.commit()
print("Customers inserted ✅")

Customers inserted ✅


In [63]:
cursor.execute("TRUNCATE TABLE fact_sales")
conn.commit()
print("Table cleared ✅")

Table cleared ✅


In [65]:
# Fix date format
orders['ord_date'] = pd.to_datetime(orders['ord_date'], errors='coerce').dt.date

# Fix amount (ensure numeric)
orders['ord_amt'] = pd.to_numeric(orders['ord_amt'], errors='coerce')

# Remove invalid rows
orders.dropna(inplace=True)

C:\Users\Hp\AppData\Local\Temp\ipykernel_21980\1938733500.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders['ord_date'] = pd.to_datetime(orders['ord_date'], errors='coerce').dt.date


In [66]:
print(orders['ord_date'].head(10))

0    2021-01-01
1    0001-01-02
2    0001-01-03
3    0001-01-06
4    0001-01-02
5    0001-02-03
6    2021-03-06
Name: ord_date, dtype: object


In [68]:
orders['ord_date'] = pd.to_datetime(orders['ord_date'], errors='coerce')

In [69]:
print(orders[orders['ord_date'].dt.year < 2000])

   ord_id  cust_id  ord_amt    prd_cat ord_date ord_time
1     102        2     4000    laptops  1-01-02     8 AM
2     103       66      900      books  1-01-03     7 AM
3     105        1    70300  jewellery  1-01-06     1900
4     112        2     4000    clothes  1-01-02     2100
5     123       66      880      books  1-02-03     4 PM


In [70]:
orders['ord_date'] = pd.to_datetime(orders['ord_date'], errors='coerce')

In [71]:
print(orders['ord_date'])

0   2021-01-01
1      1-01-02
2      1-01-03
3      1-01-06
4      1-01-02
5      1-02-03
6   2021-03-06
Name: ord_date, dtype: datetime64[s]


In [72]:
orders = orders[orders['ord_date'].dt.year >= 2000]

In [73]:
print(orders['ord_date'])

0   2021-01-01
6   2021-03-06
Name: ord_date, dtype: datetime64[s]


In [74]:
orders = orders.reset_index(drop=True)

In [77]:
for i, row in orders.iterrows():
    try:
        cursor.execute(
            """
            INSERT INTO fact_sales 
            (order_id, customer_id, product_id, amount, order_date)
            VALUES (%s, %s, %s, %s, %s)
            """,
            (
                int(row['ord_id']),
                int(row['cust_id']),
                str(row['prd_cat']),
                float(row['ord_amt']),
                row['ord_date']
            )
        )
    except Exception as e:
        print("Error at row:", i, e)

conn.commit()

In [79]:
print(orders.columns)

Index(['ord_id', 'cust_id', 'ord_amt', 'prd_cat', 'ord_date', 'ord_time'], dtype='str')


In [82]:
for i, row in orders.iterrows():
    try:
        cursor.execute(
            "INSERT INTO fact_sales (order_id, customer_id, product_id, amount, order_date) VALUES (%s, %s, %s, %s, %s)",
            (
                row['ord_id'],
                row['cust_id'],
                row['prd_cat'],
                row['ord_amt'],
                row['ord_date']
            )
        )
    except Exception as e:
        print("Skipping:", row['ord_id'], e)

conn.commit()

Skipping: 101 1062 (23000): Duplicate entry '101' for key 'fact_sales.PRIMARY'
Skipping: 135 1062 (23000): Duplicate entry '135' for key 'fact_sales.PRIMARY'


In [83]:
orders = orders.drop_duplicates(subset=['ord_id'])

In [84]:
print(orders.isnull().sum())

ord_id      0
cust_id     0
ord_amt     0
prd_cat     0
ord_date    0
ord_time    0
dtype: int64


In [85]:
orders = orders.dropna()

In [86]:
print("Inserting:", row['ord_id'])

Inserting: 135


In [87]:
query = """
INSERT INTO fact_sales (order_id, customer_id, product_id, amount, order_date)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
customer_id = VALUES(customer_id),
product_id = VALUES(product_id),
amount = VALUES(amount),
order_date = VALUES(order_date)
"""

cursor.execute(query, (
    row['ord_id'],
    row['cust_id'],
    row['prd_cat'],
    row['ord_amt'],
    row['ord_date']
))

In [88]:
row['ord_amt'] = 9999

In [89]:
print(orders.isnull().sum())
print(orders.dtypes)

ord_id      0
cust_id     0
ord_amt     0
prd_cat     0
ord_date    0
ord_time    0
dtype: int64
ord_id              int64
cust_id             int64
ord_amt             int64
prd_cat               str
ord_date    datetime64[s]
ord_time              str
dtype: object


In [90]:
print("Rows processed:", len(orders))

Rows processed: 2


In [91]:
dim_customer = orders[['cust_id']].drop_duplicates()

In [92]:
dim_product = orders[['prd_cat']].drop_duplicates()

In [93]:
dim_date = orders[['ord_date']].drop_duplicates()

In [96]:
query = """
INSERT INTO dim_customer (customer_id)
VALUES (%s)
ON DUPLICATE KEY UPDATE customer_id = customer_id
"""